# Notebook Objective - process netcdf files to include datetime at location
___
## Step-by-Step explanation of code

1.	Import Libraries and Define Functions:
•	Import necessary libraries: xarray, pandas, datetime, and os.
•	Define helper functions to get the start year, hypothesis, start date based on release, and to convert time values (T) to actual dates.
	
2.	File Paths and Ensure Output Directory Exists:
•	Define paths for input data (repo_path) and output data (output_path).
•	Ensure the output directory exists or create it.
	
3.	Define Simulations and Locations:
•	List of simulation IDs and locations to process.
	
4.	Define Functions:
•	get_start_year(sim_id): Returns the start year based on the simulation ID.
•	get_hypothesis(sim_id): Returns the hypothesis being tested based on the simulation ID.
•	get_start_date(year, release): Calculates the start date based on the start year and release value.
•	convert_to_dates(ds, start_year): Converts T values to actual dates for each particle.
•	determine_stage(t): Converts T values to life stages: embryo = 0; larvae = 1; calytopsis = 2; furcilia = 3; juvenile = 4.
	
6.	Process Each Simulation and Location:
•	Loop through each simulation ID and location.
•	Check if the corresponding netCDF file exists.
•	If the file exists:
	1.	Open the data using xarray and load it into memory.
	2.	Assign coordinates N and T.
	3.	Append metadata (start year and hypothesis) to the dataset.
	4.	Convert T values to actual dates using the determined start year.
	5.	Create a DataArray with dimensions ['N', 'T'] containing the dates and add it to the dataset as a new coordinate.
	6.	Create a DataFrame for the drifter trajectories with actual dates.
	7.	Save the DataFrame to a new CSV file.
	8.	Print a confirmation message. If the file does not exist, print a warning message.
___

In [1]:
import xarray as xr
import pandas as pd
from datetime import datetime, timedelta
import os

from shapely.geometry import Point, Polygon as ShapelyPolygon
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
# import dask
# from dask.distributed import Client, progress
import logging

# Set up logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# # Set up Dask client
# client = Client()

In [3]:
# File Paths to data
repo_path = '/Users/zephyrsylvester/repos/circumpolar-connectivity-analysis/data/'
output_path = '/Users/zephyrsylvester/repos/circumpolar-connectivity-analysis/processed_data/'

# # Ensure output directory exists
# os.makedirs(output_path, exist_ok=True)

# List of simulations and locations
sim_list = ['007', '008', '009', '010', '011', '012', '013', '014', '015', '016', '017', '018']
locations = ['BS', 'GERL', 'GP', 'MB2']

# # Log initial setup details
# logger.info(f'Initialized Dask client with dashboard available at: {client.dashboard_link}')
# logger.info(f'Processing simulations: {sim_list}')
# logger.info(f'Locations: {locations}')

In [4]:
# Function to get start year based on simulation ID
def get_start_year(sim_id):
    if sim_id in ['007', '008', '013', '015']:
        return 2016
    elif sim_id in ['010', '009', '014', '016']:
        return 2017
    elif sim_id in ['012', '011', '017', '018']:
        return 2018
    return None

# Function to get hypothesis based on simulation ID
def get_hypothesis(sim_id):
    hypotheses = {
        'h_null': ['007', '010', '012'],
        'h_ice': ['008', '009', '011'],
        'h_dvm': ['013', '014', '017'],
        'h_size': ['015', '016', '018']
    }
    for key, ids in hypotheses.items():
        if sim_id in ids:
            return key
    return None

# Function to get the start date based on start year and release value
def get_start_date(year, release):
    base_date = datetime(year, 11, 1)
    release_date = base_date + timedelta(days=int(release) * 14)  # Convert release to int and 2 weeks interval
    return release_date

# Convert `T` values to actual dates for each particle
def convert_to_dates(ds, start_year):
    release_dates = [get_start_date(start_year, r) for r in ds['release'].values]
    all_dates = []
    for release_date in release_dates:
        dates = [release_date + timedelta(days=int(t)) for t in ds['T'].values]  # Convert T to int
        all_dates.append(dates)
    return all_dates

# Function to determine the stage based on T value
def determine_stage(t):
    if 0 <= t <= 5:
        return 0
    elif 6 <= t <= 22:
        return 1
    elif 23 <= t <= 60:
        return 2
    elif 61 <= t <= 96:
        return 3
    elif t > 96:
        return 4
    return None

# Function to generate a unique larval ID
def generate_larval_ids(hypothesis, start_year, drifter_ids, location):
    hypothesis_codes = {'h_null': '00', 'h_ice': '01', 'h_dvm': '02', 'h_size': '03'}
    year_codes = {2016: '16', 2017: '17', 2018: '18'}
    location_codes = {'BS': '1', 'GERL': '2', 'GP': '3', 'MB2': '4'}
    try:
        prefix = f"{hypothesis_codes[hypothesis]}_{year_codes[start_year]}_{location_codes[location]}"
    except KeyError as e:
        raise ValueError(f"Invalid key encountered: {e}")
    unique_ids = {drifter_id: f"{prefix}_{str(index + 1).zfill(4)}" for index, drifter_id in enumerate(drifter_ids)}
    return unique_ids

In [5]:
# Process each simulation and location
def process_simulation_location(sim, location):
    subset = f'r{sim}.{location}.drifters.180d.nc'
    file = os.path.join(repo_path, subset)
    
    if os.path.exists(file):
        # Open data with xarray and load it into memory
        ds = xr.open_dataset(file).load()

        # Assign Coordinates
        ds = ds.assign_coords(N=ds['N'], T=ds['T'])

        # Append metadata
        start_year = get_start_year(sim)
        hypothesis = get_hypothesis(sim)
        logger.info(f"Processing simulation {sim} with start year {start_year} and hypothesis {hypothesis}")
        if start_year is None or hypothesis is None:
            logger.warning(f"Invalid start year or hypothesis for simulation {sim}")
            return None
        ds = ds.assign_attrs(start_year=start_year, hypothesis=hypothesis)

        # Convert to actual dates using the determined start year
        all_dates = convert_to_dates(ds, start_year)

        # Create a DataArray with dimensions ['N', 'T'] containing the dates
        date_array = xr.DataArray(all_dates, dims=['N', 'T'])

        # Add the dates DataArray to the dataset as a new coordinate
        ds = ds.assign_coords(date=date_array)

        # Generate unique larval IDs
        drifter_ids = ds['N'].values
        try:
            unique_larval_ids = generate_larval_ids(hypothesis, start_year, drifter_ids, location)
        except ValueError as e:
            logger.error(e)
            return None

        # Create a DataFrame for the drifter trajectories with actual dates
        trajectories = [
            {
                'N': n, 'T': t,
                'date': ds['date'].sel(N=n, T=t).values,
                'lat': ds['lat'].sel(N=n, T=t).values,
                'lon': ds['lon'].sel(N=n, T=t).values,
                'release': ds['release'].sel(N=n).values,
                'stage': determine_stage(t),
                'larval_id': unique_larval_ids[n],
                'start_year': start_year,
                'hypothesis': hypothesis,
                'IDL_loc': location
            }
            for n in ds['N'].values
            for t in ds['T'].values
        ]

        trajectories_df = pd.DataFrame(trajectories)

        # Save the DataFrame to a CSV file
        output_file = os.path.join(output_path, f'proc_traj_{location}_{start_year}_{hypothesis}.csv')
        trajectories_df.to_csv(output_file, index=False)
        logger.info(f"Processed and saved: {output_file}")
    else:
        logger.warning(f"File not found: {file}")

In [6]:
# Process all simulations and locations
for sim_id in sim_list:
    for location in locations:
        process_simulation_location(sim_id, location)

INFO:__main__:Processing simulation 007 with start year 2016 and hypothesis h_null
INFO:__main__:Processed and saved: /Users/zephyrsylvester/repos/connectivity-manuscript/processed_data/proc_traj_BS_2016_h_null.csv
INFO:__main__:Processing simulation 007 with start year 2016 and hypothesis h_null
INFO:__main__:Processed and saved: /Users/zephyrsylvester/repos/connectivity-manuscript/processed_data/proc_traj_GERL_2016_h_null.csv
INFO:__main__:Processing simulation 007 with start year 2016 and hypothesis h_null
INFO:__main__:Processed and saved: /Users/zephyrsylvester/repos/connectivity-manuscript/processed_data/proc_traj_GP_2016_h_null.csv
INFO:__main__:Processing simulation 007 with start year 2016 and hypothesis h_null
INFO:__main__:Processed and saved: /Users/zephyrsylvester/repos/connectivity-manuscript/processed_data/proc_traj_MB2_2016_h_null.csv
INFO:__main__:Processing simulation 008 with start year 2016 and hypothesis h_ice
INFO:__main__:Processed and saved: /Users/zephyrsylvest

In [7]:
print('done')

done
